In [7]:
import pandas as pd

import os

In [ ]:
base_paths = {
    1: 'equal_satisfaction/individual_user/group_01.tsv',
    2: 'equal_satisfaction/group_user/group_02.tsv',
    3: 'envy_freeness/individual_user/group_03.tsv',
    4: 'rawlsian/individual_user/group_04.tsv',
    5: 'utilitarian/individual_user/group_05.tsv',
    7: 'ued/individual_provider/group_07.tsv',
    8: 'ued/group_item/group_08.tsv',
    9: 'qwe/individual_provider/group_09.tsv',
    11: 'rawlsian/individual_provider/group_11.tsv',
    12: 'rawlsian/individual_item/group_12.tsv',
}

base_path_root = r'results'

dfs = {}

for group, path in base_paths.items():
    df_path = f'{base_path_root}/{path}'
    df = pd.read_csv(df_path, sep='\t')
    dfs[group] = df

In [9]:
dfs.keys()

dict_keys([1, 2, 3, 4, 5, 7, 8, 9, 11, 12])

# Organizing results in processed_results/

In [10]:
def get_last_run(df):
    groups = df.groupby(['dataset', 'seed', 'run', 'k', 'method']).groups
    remain_indexes = []
    for name, group in groups.items():
        remain_indexes.append(group.to_list()[-1])
    return df.loc[remain_indexes]

def processing_df(path):
    df = pd.read_csv(path, sep='\t')
    df = df[df['dataset'] != 'black_friday_implicit']
    df = get_last_run(df)
    return df

def make_processed_results():
    dfs = {}
    for group, path in base_paths.items():
        df_path = f'{base_path_root}/{path}'
        df = processing_df(df_path)
        dfs[group] = df
        print(f'{group}: {path}')
        print(f'Number of rows should be: {len(df['method'].unique()) * len(df['dataset'].unique()) * len(df['seed'].unique()) * len(df['k'].unique())}\nCurrent number of rows: {len(df)}\n\n')

        os.makedirs(f'{base_path_root}/processed_results', exist_ok=True)
        #dfs[group] = df
        df.to_csv(f'{base_path_root}/processed_results/group_{group}.tsv', sep='\t', index=False)

In [12]:
path = f'{base_path_root}/{base_paths[2]}'
processing_df(path).head()

,method,dataset,seed,run,k,ndcg_baseline,ndcg_fair_method,stakeholder,granularity,fairness_metric_name,fairness_metric_baseline,fairness_metric
205,CPFair,amazon,42,0,10,0.432582,0.225307,user,group,variance_group_ndcg,0.033493,0.000857
245,CPFair,amazon,42,0,25,0.459825,0.251170,user,group,variance_group_ndcg,0.029454,0.000221
285,CPFair,amazon,42,0,50,0.470432,0.259892,user,group,variance_group_ndcg,0.027546,0.000090
325,CPFair,amazon,42,0,100,0.480537,0.267156,user,group,variance_group_ndcg,0.025853,0.000020
365,CPFair,amazon,42,0,150,0.485167,0.269810,user,group,variance_group_ndcg,0.025179,0.000006


In [13]:
make_processed_results()

1: equal_satisfaction/individual_user/group_01.tsv
Number of rows should be: 600
Current number of rows: 600


2: equal_satisfaction/group_user/group_02.tsv
Number of rows should be: 200
Current number of rows: 200


3: envy_freeness/individual_user/group_03.tsv
Number of rows should be: 200
Current number of rows: 200


4: rawlsian/individual_user/group_04.tsv
Number of rows should be: 600
Current number of rows: 600


5: utilitarian/individual_user/group_05.tsv
Number of rows should be: 200
Current number of rows: 200


7: ued/individual_provider/group_07.tsv
Number of rows should be: 400
Current number of rows: 400


8: ued/group_item/group_08.tsv
Number of rows should be: 200
Current number of rows: 200


9: qwe/individual_provider/group_09.tsv
Number of rows should be: 400
Current number of rows: 400


11: rawlsian/individual_provider/group_11.tsv
Number of rows should be: 200
Current number of rows: 200


12: rawlsian/individual_item/group_12.tsv
Number of rows should be: 400
Cur

# Agg mean and std of metrics

In [14]:
mean_ndcg_150 = pd.DataFrame(columns=['dataset', 'method', 'mean_ndcg'])

for group_number, df in dfs.items():
    fairness_metric_name = df['fairness_metric_name'].unique()[0]
    print(fairness_metric_name)
    df = df[df['dataset'] != 'black_friday_implicit']
    agg_df = df.groupby(['dataset', 'k', 'method']).agg(
        p25_ndcg_baseline=('ndcg_baseline', lambda x: x.quantile(0.25)),
        mean_ndcg_baseline=('ndcg_baseline', 'mean'),
        p75_ndcg_baseline=('ndcg_baseline', lambda x: x.quantile(0.75)),
        std_ndcg_baseline=('ndcg_baseline', 'std'),
        p25_ndcg_method=('ndcg_fair_method', lambda x: x.quantile(0.25)),
        mean_ndcg_method=('ndcg_fair_method', 'mean'),
        p75_ndcg_method=('ndcg_fair_method', lambda x: x.quantile(0.75)),
        std_ndcg_method=('ndcg_fair_method', 'std'),
        p25_fairness_baseline=('fairness_metric_baseline', lambda x: x.quantile(0.25)),
        mean_fairness_baseline=('fairness_metric_baseline', 'mean'),
        p75_fairness_baseline=('fairness_metric_baseline', lambda x: x.quantile(0.75)),
        std_fairness_baseline=('fairness_metric_baseline', 'std'),
        p25_fairness_method=('fairness_metric', lambda x: x.quantile(0.25)),
        mean_fairness_method=('fairness_metric', 'mean'),
        p75_fairness_method=('fairness_metric', lambda x: x.quantile(0.75)),
        std_fairness_method=('fairness_metric', 'std'),
    ).reset_index()

    baseline_ndcg_150 = agg_df[agg_df['k'] == 150][['dataset', 'mean_ndcg_baseline']].drop_duplicates()
    baseline_ndcg_150['method'] = 'baseline'
    baseline_ndcg_150 = baseline_ndcg_150.rename(columns={'mean_ndcg_baseline': 'mean_ndcg'})

    method_ndcg_150 = agg_df[agg_df['k'] == 150][['dataset', 'method', 'mean_ndcg_method']].rename(
        columns={'mean_ndcg_method': 'mean_ndcg'}
    )

    ndcg_150_combined = pd.concat([baseline_ndcg_150[['dataset', 'method', 'mean_ndcg']], method_ndcg_150], ignore_index=True)

    mean_ndcg_150 = pd.concat([mean_ndcg_150, ndcg_150_combined], ignore_index=True)
    
    os.makedirs(f'{base_path_root}/agg_results', exist_ok=True)

    agg_df.to_csv(f'{base_path_root}/agg_results/group_{group_number}_aggregated.tsv', sep='\t', index=False)

variance_ndcg
variance_group_ndcg
envy_freeness
worse_off_cumulative_utility


/tmp/ipykernel_32159/776214082.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  mean_ndcg_150 = pd.concat([mean_ndcg_150, ndcg_150_combined], ignore_index=True)


sum_user_utilities
variance_provider_exposure
variance_item_group_exposure
provider_exposure_relevance_ratio_variance
worse_off_cumulative_provider_exposure
worse_off_cumulative_item_exposure


# RQ1: mean NDCG for each method for each dataset

In [15]:
rq1_df = mean_ndcg_150.groupby(['dataset', 'method'])['mean_ndcg'].mean().reset_index()
rq1_df

,dataset,method,mean_ndcg
0,amazon,Ada2Fair,0.299012
1,amazon,CPFair,0.168153
2,amazon,FairRec,0.146539
3,amazon,FairSort,0.493676
4,amazon,GGF,0.083999
5,amazon,LeadFairRec,0.104719
6,amazon,SEAL,0.137782
7,amazon,TFLD,0.334298
8,amazon,TFROM,0.037170
9,amazon,baseline,0.494623


# RQ1 -- Adjusting for LaTeX format

In [16]:
methods_to_num = {
    'baseline': 0,
    'CPFair': 1,
    'LeadFairRec': 2,
    'FairSort': 3,
    'TFROM': 4,
    'Ada2Fair': 5,
    'FairRec': 6,
    'GGF': 7,
    'SEAL': 8,
    'TFLD': 9,
}

datasets_to_num = {
    'amazon': 0,
    'ambar': 1,
    'movielens': 2,
    'yelp': 3,
}

rq1_df.replace({'method': methods_to_num, 'dataset': datasets_to_num}, inplace=True)

/tmp/ipykernel_32159/3652718277.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rq1_df.replace({'method': methods_to_num, 'dataset': datasets_to_num}, inplace=True)


In [17]:
rq1_df = rq1_df[['method', 'dataset', 'mean_ndcg']].sort_values(by=['dataset', 'method']).reset_index(drop=True)

In [19]:
rq1_df.to_latex(index=False, header=False, float_format="%.4f", buf=f'{base_path_root}/agg_results/rq1_df.tex')

# RQ4: are there situations in which two-sided methods fail to improve fairness for one side?

Six methods were evaluated on **both** a consumer-side and a provider/item-side
fairness metric. The unit of analysis is **method x stakeholder side**: for each one we
ask whether that side's fairness metric *consistently* improves over the baseline,
across all four datasets.

**Answer.** Of the six methods, exactly one (TFROM) consistently improves both sides.
Two (GGF, TFLD) consistently improve the provider side while consistently *degrading*
the consumer side. Three are not consistent on at least one side.

Three things about the data drive the design of this section:

1. **Relative improvement is unusable.** `(baseline - method) / baseline` reaches
   -10058 (CPFair/ambar) and -91213 (CPFair/yelp) because some baselines are near
   zero, and 21 rows at k=150 have a baseline of exactly 0. We report the
   **normalized fairness gain** instead, dividing by the *sum* rather than the
   baseline, which is bounded in [-1, +1] and comparable across every metric.
2. **A win count carries no uncertainty and hides magnitude.** `1/10 seeds improved`
   reads as a strong degradation; on the normalized scale that cell (FairSort/amazon,
   consumer) is `-0.01 [-0.01, -0.00]` -- real, but negligible. Every estimate here
   carries a distribution-free confidence interval.
3. **The choice of test barely matters, and n=10 is the real constraint.** 40 of the
   48 per-configuration cells are saturated at 0/10 or 10/10 with the effect 7-265x
   the seed-to-seed SD, so every exact paired test returns its floor of
   `2/2**10 = 0.00195`. The sign test and Wilcoxon agree on 23 of 24 configurations.
   We use the **exact sign test** because it is invariant to monotone rescaling, so
   the verdict cannot depend on the scale the figures happen to display, and its
   order-statistic CI is invariant in the same way.

LeadFairRec, FairRec and SEAL are **excluded**: only their consumer side was measured
in this run (groups 6 and 10 are commented out in
`experiments/grouped_experiments.py`, MultiFR never ran). Their absence is a gap in
coverage, not evidence that they do not fail.

In [ ]:
import numpy as np
from scipy.stats import binom, binomtest, wilcoxon

# Fairness metrics disagree on direction and no column says so, so write it down.
# Variances and mean-average-envy are better small; welfare sums are better large.
LOWER_IS_BETTER = {
    'variance_ndcg',
    'variance_group_ndcg',
    'variance_provider_exposure',
    'variance_item_group_exposure',
    'provider_exposure_relevance_ratio_variance',
    'envy_freeness',  # this is *mean average envy* (experiments/metrics.py:265)
}

# The six methods measured on both sides: method -> (consumer metric, provider metric).
PAIRS = {
    'TFROM':    ('variance_ndcg',                'variance_provider_exposure'),
    'FairSort': ('variance_ndcg',                'variance_provider_exposure'),
    'CPFair':   ('variance_group_ndcg',          'variance_item_group_exposure'),
    'Ada2Fair': ('worse_off_cumulative_utility', 'worse_off_cumulative_provider_exposure'),
    'TFLD':     ('worse_off_cumulative_utility', 'worse_off_cumulative_item_exposure'),
    'GGF':      ('worse_off_cumulative_utility', 'worse_off_cumulative_item_exposure'),
}

METHOD_ORDER = ['TFROM', 'FairSort', 'CPFair', 'Ada2Fair', 'TFLD', 'GGF']
DATASETS = ['amazon', 'ambar', 'movielens', 'yelp']
SIDES = ['consumer', 'provider']
RQ4_K = 150
ALPHA = 0.05

# One long frame over every group file, so a cell can be looked up by metric name.
rq4_raw = pd.concat(
    [pd.read_csv(f'{base_path_root}/processed_results/group_{g}.tsv', sep='\t')
     for g in base_paths],
    ignore_index=True,
)
rq4_raw.shape

In [ ]:
def nfg(df, method, dataset, metric, k):
    """Normalized fairness gain, per seed, in [-1, +1]. Positive always means fairer.

    Dividing by the *sum* rather than the baseline is what keeps this readable: the
    ordinary relative change reaches -91213 on CPFair/yelp because some baselines are
    almost zero. All fairness values here are non-negative, so the ratio is bounded.
    The `where` guard covers baseline == method == 0, i.e. "no change".
    """
    cell = df[(df['method'] == method)
              & (df['dataset'] == dataset)
              & (df['fairness_metric_name'] == metric)
              & (df['k'] == k)].sort_values('seed')
    baseline = cell['fairness_metric_baseline'].to_numpy()
    fair = cell['fairness_metric'].to_numpy()
    delta = (baseline - fair) if metric in LOWER_IS_BETTER else (fair - baseline)
    total = baseline + fair
    return np.divide(delta, total, out=np.zeros_like(delta, dtype=float), where=total > 0)


def median_ci(x, alpha=ALPHA):
    """Order-statistic CI for the median, plus the coverage it actually achieves.

    Distribution-free and invariant to monotone rescaling, so it can never contradict
    the sign test. Coverage is computed rather than assumed: at n=10 no two-sided
    interval lands on 95% -- the neighbours are 89.1% and 97.9%, and we take the
    conservative one.
    """
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    chosen = None
    for j in range(n // 2):
        coverage = 1 - 2 * binom.cdf(j, n, 0.5)
        if coverage >= 1 - alpha:
            chosen = (j, coverage)
        else:
            break
    if chosen is None:                       # n too small for any valid interval
        return float(np.median(x)), float(x[0]), float(x[-1]), 0.0
    j, coverage = chosen
    return float(np.median(x)), float(x[j]), float(x[n - 1 - j]), float(coverage)


def sign_test(x):
    """Exact two-sided sign test, ties dropped. Returns (seeds improved, n used, p)."""
    nonzero = x[x != 0]
    wins = int((x > 0).sum())
    if len(nonzero) == 0:
        return wins, 0, 1.0
    return wins, len(nonzero), float(binomtest((nonzero > 0).sum(), len(nonzero), 0.5).pvalue)


def benjamini_hochberg(pvals):
    """BH false-discovery-rate q-values (statsmodels-free, ~8 lines).

    Holm is unusable here: with 10 seeds the smallest attainable exact p is
    2/2**10 = 0.00195, and Holm's 48x multiplier pushes every cell above 0.05 --
    a power artefact, not a null result.
    """
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    q = np.empty(n)
    running = 1.0
    for rank in range(n - 1, -1, -1):
        i = order[rank]
        running = min(running, pvals[i] * n / (rank + 1))
        q[i] = running
    return q


def rq4_table(df, k, pairs=PAIRS):
    """Per-configuration table for one cutoff k, BH applied across all 2 x n_cells tests."""
    rows = []
    for method, metrics in pairs.items():
        for dataset in DATASETS:
            row = {'method': method, 'dataset': dataset}
            for side, metric in zip(SIDES, metrics):
                x = nfg(df, method, dataset, metric, k)
                assert len(x) == 10, f'{method}/{dataset}/{metric} at k={k}: {len(x)} seeds'
                assert np.all(np.abs(x) <= 1 + 1e-12), f'NFG out of range: {method}/{dataset}'
                wins, _, p = sign_test(x)
                med, lo, hi, coverage = median_ci(x)
                row[f'{side}_metric'] = metric
                row[f'{side}_wins'] = wins
                row[f'{side}_nfg'], row[f'{side}_lo'], row[f'{side}_hi'] = med, lo, hi
                row[f'{side}_coverage'] = coverage
                row[f'{side}_p'] = p
                # kept only so the robustness cell can show the two tests agree
                row[f'{side}_wilcoxon_p'] = wilcoxon(x).pvalue if (x != 0).any() else 1.0
            rows.append(row)

    out = pd.DataFrame(rows)
    q = benjamini_hochberg(np.concatenate([out['consumer_p'], out['provider_p']]))
    out['consumer_q'], out['provider_q'] = q[:len(out)], q[len(out):]
    qw = benjamini_hochberg(np.concatenate([out['consumer_wilcoxon_p'],
                                            out['provider_wilcoxon_p']]))
    out['consumer_wilcoxon_q'], out['provider_wilcoxon_q'] = qw[:len(out)], qw[len(out):]

    # A side improved only if it is significant AND in the right direction.
    for side in SIDES:
        out[f'{side}_better'] = (out[f'{side}_q'] < ALPHA) & (out[f'{side}_wins'] > 5)
        out[f'{side}_worse'] = (out[f'{side}_q'] < ALPHA) & (out[f'{side}_wins'] < 5)
    out['verdict'] = np.select(
        [out['consumer_better'] & out['provider_better'],
         out['consumer_better'],
         out['provider_better']],
        ['both sides', 'consumer only', 'provider only'],
        default='neither side')
    return out

In [ ]:
# Per-configuration detail. Built first because the consistency verdict below counts
# how many datasets each side improves or degrades on.
rq4_df = rq4_table(rq4_raw, RQ4_K)

print('per-cell CI coverage:', sorted({*rq4_df['consumer_coverage'], *rq4_df['provider_coverage']}))
rq4_df[['method', 'dataset',
        'consumer_wins', 'consumer_nfg', 'consumer_lo', 'consumer_hi', 'consumer_q',
        'provider_wins', 'provider_nfg', 'provider_lo', 'provider_hi', 'provider_q',
        'verdict']]

## RQ4 Table 1 (primary): does each stakeholder side consistently improve?

One row per **method x stakeholder side**. `NFG` is the normalized fairness gain
pooled over all 4 datasets x 10 seeds = 40 paired runs, with its order-statistic
confidence interval. A side is *consistently improved* only if it improves
significantly on **all four** datasets, and *consistently degraded* only if it
worsens significantly on all four; anything else is **mixed**.

This is the table that answers the research question.

In [ ]:
rows = []
for method, metrics in PAIRS.items():
    for side, metric in zip(SIDES, metrics):
        # Pooling the four datasets is only legitimate because NFG is unit-free.
        pooled = np.concatenate([nfg(rq4_raw, method, d, metric, RQ4_K) for d in DATASETS])
        wins, _, p = sign_test(pooled)
        med, lo, hi, coverage = median_ci(pooled)
        cells = rq4_df[rq4_df['method'] == method]
        up, down = int(cells[f'{side}_better'].sum()), int(cells[f'{side}_worse'].sum())
        rows.append({
            'method': method, 'side': side, 'metric': metric,
            'nfg': med, 'lo': lo, 'hi': hi, 'coverage': coverage,
            'wins': wins, 'n': len(pooled), 'p': p,
            'datasets_up': up, 'datasets_down': down,
            'consistency': ('consistently improves' if up == len(DATASETS)
                            else 'consistently degrades' if down == len(DATASETS)
                            else 'mixed / not consistent'),
        })

rq4_consistency = pd.DataFrame(rows)
rq4_consistency['q'] = benjamini_hochberg(rq4_consistency['p'])
rq4_consistency['method'] = pd.Categorical(rq4_consistency['method'], METHOD_ORDER, ordered=True)
rq4_consistency = rq4_consistency.sort_values(['method', 'side']).reset_index(drop=True)
rq4_consistency.to_csv(f'{base_path_root}/agg_results/rq4_consistency.tsv', sep='\t', index=False)

improves_both = [m for m in METHOD_ORDER
                 if (rq4_consistency.loc[rq4_consistency['method'] == m, 'consistency']
                     == 'consistently improves').all()]
print(f'methods consistently improving BOTH sides: {improves_both} '
      f'(of {len(PAIRS)} measured on both sides)')
print(f'pooled CI coverage: {rq4_consistency["coverage"].iloc[0]:.3f}')

rq4_consistency[['method', 'side', 'nfg', 'lo', 'hi', 'wins', 'n', 'q',
                 'datasets_up', 'datasets_down', 'consistency']]

In [ ]:
def stars(q):
    return '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else 'n.s.'


cov = rq4_consistency['coverage'].iloc[0]
# A bare '%' in a column name silently comments out the rest of the LaTeX line.
cov_tex = f'{cov:.0%}'.replace('%', r'\%')

rq4_consistency_tex = pd.DataFrame({
    'Method': rq4_consistency['method'],
    'Side': rq4_consistency['side'],
    f'NFG [{cov_tex} CI]': [f'{m:+.2f} [{l:+.2f}, {h:+.2f}]' for m, l, h
                            in zip(rq4_consistency['nfg'], rq4_consistency['lo'],
                                   rq4_consistency['hi'])],
    'Runs improved': rq4_consistency['wins'].astype(str) + '/' + rq4_consistency['n'].astype(str),
    'Sig.': rq4_consistency['q'].map(stars),
    'Datasets $\\uparrow$': rq4_consistency['datasets_up'].astype(str) + '/4',
    'Datasets $\\downarrow$': rq4_consistency['datasets_down'].astype(str) + '/4',
    'Consistency': rq4_consistency['consistency'],
})
rq4_consistency_tex.to_latex(index=False,
                             buf=f'{base_path_root}/agg_results/rq4_consistency.tex')

print('NFG = normalized fairness gain, (baseline - method)/(baseline + method), positive = fairer.')
print(f'CI = order-statistic interval for the median, exact coverage {cov:.1%}.')
print('Sig. = exact sign test over 40 paired runs, BH-adjusted over the 12 rows.')
rq4_consistency_tex

In [ ]:
# The same evidence counted per configuration, at escalating strengths of claim.
# "Failed to improve" includes cells with no significant change; "actively degraded"
# requires a significant move in the wrong direction; the third row is the two-sided
# trade-off the research question is really about.
c_better, p_better = rq4_df['consumer_better'], rq4_df['provider_better']
c_worse, p_worse = rq4_df['consumer_worse'], rq4_df['provider_worse']

rq4_claims = pd.DataFrame([
    ('improved both sides', int((c_better & p_better).sum())),
    ('failed to improve at least one side', int((~(c_better & p_better)).sum())),
    ('actively made at least one side less fair', int((c_worse | p_worse).sum())),
    ('improved one side at the expense of the other',
     int(((c_worse & p_better) | (p_worse & c_better)).sum())),
    ('made both sides less fair', int((c_worse & p_worse).sum())),
], columns=['claim', 'configurations'])
rq4_claims['of'] = len(rq4_df)
rq4_claims.to_csv(f'{base_path_root}/agg_results/rq4_claim_summary.tsv', sep='\t', index=False)
rq4_claims

In [ ]:
rq4_df.to_csv(f'{base_path_root}/agg_results/rq4_main.tsv', sep='\t', index=False)

# Supporting table: the same 12 rows broken out per dataset, so a reader can see
# *where* a "mixed" verdict in Table 1 comes from. wins stays in the .tsv as raw
# evidence but leaves the paper table -- NFG plus its interval says more.
cell_cov = rq4_df['consumer_coverage'].iloc[0]
cell_cov_tex = f'{cell_cov:.0%}'.replace('%', r'\%')   # bare % comments out the LaTeX line


def effect(nfg_value, lo, hi, q):
    """Sign, interval and significance in one cell.

    They must stay together: a bare '**' in its own column beside NFG = -0.87 reads
    as 'significantly better' when it means the opposite.
    """
    mark = '***' if q < 0.001 else '**' if q < 0.01 else '*' if q < 0.05 else ''
    return f'{nfg_value:+.2f} [{lo:+.2f}, {hi:+.2f}]{mark}'


rq4_tex = pd.DataFrame({
    'Method': rq4_df['method'],
    'Dataset': rq4_df['dataset'],
    f'Consumer NFG [{cell_cov_tex} CI]': [effect(*z) for z in zip(
        rq4_df['consumer_nfg'], rq4_df['consumer_lo'], rq4_df['consumer_hi'], rq4_df['consumer_q'])],
    f'Provider NFG [{cell_cov_tex} CI]': [effect(*z) for z in zip(
        rq4_df['provider_nfg'], rq4_df['provider_lo'], rq4_df['provider_hi'], rq4_df['provider_q'])],
    'Improved': rq4_df['verdict'],
})
rq4_tex['Method'] = pd.Categorical(rq4_tex['Method'], METHOD_ORDER, ordered=True)
rq4_tex = rq4_tex.sort_values(['Method', 'Dataset']).reset_index(drop=True)
rq4_tex.to_latex(index=False, buf=f'{base_path_root}/agg_results/rq4_main.tex')

print(f'per-cell CI coverage {cell_cov:.1%}; * q<.05, ** q<.01, *** q<.001 '
      f'(sign test, BH over 48 tests)')
rq4_tex

In [ ]:
# Supplementary: the raw magnitudes the main table deliberately leaves out, plus the
# accuracy cost, so a reader can see *how much* moved and not just which way.
raw_rows = []
for method, (user_metric, prov_metric) in PAIRS.items():
    for dataset in DATASETS:
        for side, metric in (('consumer', user_metric), ('provider', prov_metric)):
            cell = rq4_raw[(rq4_raw['method'] == method)
                           & (rq4_raw['dataset'] == dataset)
                           & (rq4_raw['fairness_metric_name'] == metric)
                           & (rq4_raw['k'] == RQ4_K)]
            raw_rows.append({
                'method': method, 'dataset': dataset, 'side': side, 'metric': metric,
                'direction': 'lower is better' if metric in LOWER_IS_BETTER else 'higher is better',
                'mean_fairness_baseline': cell['fairness_metric_baseline'].mean(),
                'mean_fairness_method': cell['fairness_metric'].mean(),
                'mean_ndcg_baseline': cell['ndcg_baseline'].mean(),
                'mean_ndcg_method': cell['ndcg_fair_method'].mean(),
            })

rq4_raw_means = pd.DataFrame(raw_rows)
rq4_raw_means.to_csv(f'{base_path_root}/agg_results/rq4_raw.tsv', sep='\t', index=False)
rq4_raw_means.head(8)

In [ ]:
# Robustness 1 -- is k=150 representative? Re-run the whole table at every cutoff and
# compare verdicts. Where a cell does move, it moves *toward* one-sided failure as k grows,
# so k=150 is not a cherry-pick in favour of the conclusion.
rq4_k_stability = pd.DataFrame({
    k: rq4_table(rq4_raw, k).set_index(['method', 'dataset'])['verdict']
    for k in sorted(rq4_raw['k'].unique())
})
rq4_k_stability['same_at_every_k'] = rq4_k_stability.nunique(axis=1) == 1
rq4_k_stability.to_csv(f'{base_path_root}/agg_results/rq4_k_stability.tsv', sep='\t')

print('identical verdict across all k:',
      int(rq4_k_stability['same_at_every_k'].sum()), '/', len(rq4_k_stability))
print()
print(pd.DataFrame({k: rq4_table(rq4_raw, k)['verdict'].value_counts()
                    for k in sorted(rq4_raw['k'].unique())}).fillna(0).astype(int).to_string())
rq4_k_stability

In [ ]:
# Robustness 2 -- TFROM and FairSort have a second provider-side metric (group 9,
# merit-aware exposure/relevance ratio variance). Swap it in for group 7 and check the
# verdicts do not move.
RQ4_G9_PAIRS = {
    'TFROM':    ('variance_ndcg', 'provider_exposure_relevance_ratio_variance'),
    'FairSort': ('variance_ndcg', 'provider_exposure_relevance_ratio_variance'),
}
rq4_g9 = rq4_table(rq4_raw, RQ4_K, pairs=RQ4_G9_PAIRS)
rq4_g9.to_csv(f'{base_path_root}/agg_results/rq4_robustness_g9.tsv', sep='\t', index=False)

g9_keyed = rq4_g9.set_index(['method', 'dataset'])['verdict']
g7_keyed = rq4_df.set_index(['method', 'dataset'])['verdict'].loc[g9_keyed.index]
print('group 9 agrees with group 7 on every cell:', bool((g9_keyed == g7_keyed).all()))

rq4_g9[['method', 'dataset', 'provider_wins', 'provider_nfg',
        'provider_lo', 'provider_hi', 'provider_q', 'verdict']]

In [ ]:
# Robustness 3 -- does the conclusion depend on picking the sign test over Wilcoxon?
# It does not: the two agree everywhere except one genuinely inconclusive cell, which
# is why the choice of test is a footnote in the paper rather than a section.
def verdict_from(row, q_suffix):
    better = {s: row[f'{s}{q_suffix}'] < ALPHA and row[f'{s}_wins'] > 5 for s in SIDES}
    if all(better.values()):
        return 'both sides'
    return next((f'{s} only' for s in SIDES if better[s]), 'neither side')


compare = pd.DataFrame({
    'method': rq4_df['method'], 'dataset': rq4_df['dataset'],
    'sign': rq4_df.apply(verdict_from, axis=1, q_suffix='_q'),
    'wilcoxon': rq4_df.apply(verdict_from, axis=1, q_suffix='_wilcoxon_q'),
})
compare['agree'] = compare['sign'] == compare['wilcoxon']
compare.to_csv(f'{base_path_root}/agg_results/rq4_robustness_test.tsv', sep='\t', index=False)

print(f'sign test vs Wilcoxon: agree on {int(compare["agree"].sum())}/{len(compare)} configurations')
compare[~compare['agree']]

## RQ4 figures

- **`fig_rq4_sides`** is the per-configuration overview: every method x dataset cell
  split into its consumer half and its provider half, so a one-sided failure is
  literally a half-red cell. Each half carries its NFG point estimate, so identity is
  never colour-alone.
- **`fig_rq4_forest`** is the confidence figure and the visual twin of Table 1: 12
  rows, point estimate plus interval, on one shared [-1, +1] axis. Whisker length is
  what the win count could not show -- compare FairSort's consumer bar sitting on top
  of zero against GGF's, which is nowhere near it.

Colours are the diverging pair blue/red with a neutral midpoint, validated against the
light chart surface (worst-pair delta-E 21.6 protan, 32.3 normal vision).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Diverging pair: blue = fairness improved, red = fairness degraded, neutral gray =
# no significant change. The two poles were validated against the light chart
# surface -- worst-pair delta-E 21.6 under protanopia, 32.3 for normal vision.
IMPROVED  = '#2a78d6'
DEGRADED  = '#e34948'
NO_CHANGE = '#f0efec'
SURFACE   = '#fcfcfb'
INK       = '#0b0b0b'
INK_MUTED = '#898781'
GRIDLINE  = '#e1e0d9'

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'figure.facecolor': SURFACE,
    'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE,
    'text.color': INK,
    'axes.labelcolor': INK,
    'xtick.color': INK_MUTED,
    'ytick.color': INK_MUTED,
    'axes.edgecolor': GRIDLINE,
})


def side_color(q, r):
    if q < ALPHA and r > 0:
        return IMPROVED
    if q < ALPHA and r < 0:
        return DEGRADED
    return NO_CHANGE

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.4))
GAP = 0.04                 # surface gap so adjacent fills never touch
HALF = 0.5 - GAP

for row, method in enumerate(METHOD_ORDER):
    for col, dataset in enumerate(DATASETS):
        cell = rq4_df[(rq4_df['method'] == method) & (rq4_df['dataset'] == dataset)].iloc[0]
        for j, side in enumerate(SIDES):
            color = side_color(cell[f'{side}_q'], cell[f'{side}_wins'] - 5)
            x = col + (GAP if j == 0 else 0.5 + GAP)
            ax.add_patch(Rectangle((x, row + GAP), HALF, 1 - 2 * GAP,
                                   facecolor=color, edgecolor='none'))
            ax.text(x + HALF / 2, row + 0.5, f"{cell[f'{side}_nfg']:+.2f}",
                    ha='center', va='center', fontsize=9.5,
                    color=SURFACE if color != NO_CHANGE else INK_MUTED)

ax.set_xlim(0, len(DATASETS))
ax.set_ylim(len(METHOD_ORDER), -0.02)
ax.set_xticks([])
ax.set_yticks(np.arange(len(METHOD_ORDER)) + 0.5)
ax.set_yticklabels(METHOD_ORDER, fontsize=10, color=INK)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

for col, dataset in enumerate(DATASETS):
    ax.text(col + 0.50, -0.46, dataset, ha='center', va='bottom', fontsize=10, color=INK)
    ax.text(col + 0.25, -0.16, 'consumer', ha='center', va='bottom', fontsize=7.5, color=INK_MUTED)
    ax.text(col + 0.75, -0.16, 'provider', ha='center', va='bottom', fontsize=7.5, color=INK_MUTED)

ax.set_title(f'Normalized fairness gain per stakeholder side   (k={RQ4_K}, 10 seeds)',
             fontsize=11.5, color=INK, pad=52, loc='left')
ax.text(0, 1.145, 'NFG in [-1, +1]; positive = fairer than baseline. '
                  'Intervals are in the forest plot below.',
        transform=ax.transAxes, fontsize=8.5, color=INK_MUTED, va='bottom')

handles = [Rectangle((0, 0), 1, 1, facecolor=c,
                     edgecolor=GRIDLINE if c == NO_CHANGE else 'none')
           for c in (IMPROVED, DEGRADED, NO_CHANGE)]
ax.legend(handles, ['fairer  (q < .05)', 'less fair  (q < .05)', 'no significant change'],
          loc='upper center', bbox_to_anchor=(0.5, -0.045), ncol=3, frameon=False,
          fontsize=9, handlelength=1.3, handleheight=1.1, columnspacing=1.6)

fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(f'{base_path_root}/agg_results/fig_rq4_sides.{ext}',
                dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Forest plot of Table 1: 12 rows, one per method x stakeholder side, pooled over the
# 40 paired runs. Whisker length is the confidence -- this is what the win count could
# not show. Twelve rows rather than forty-eight keeps it legible and maps 1:1 to Table 1.
from matplotlib.lines import Line2D

# The neutral midpoint used in the matrix (#f0efec) sits 1.12:1 against the surface --
# fine as a fill with a number on top, invisible as a line. Non-significant rows use
# the muted ink instead: still unmistakably "neither blue nor red", but actually legible.
NO_CHANGE_LINE = INK_MUTED

fig, ax = plt.subplots(figsize=(7.6, 5.2))

ypos, labels, seps = [], [], []
y = 0.0
for method in METHOD_ORDER:
    for side in SIDES:
        r = rq4_consistency[(rq4_consistency['method'] == method)
                            & (rq4_consistency['side'] == side)].iloc[0]
        significant = r['q'] < ALPHA
        color = (IMPROVED if r['wins'] > r['n'] / 2 else DEGRADED) if significant else NO_CHANGE_LINE
        ax.plot([r['lo'], r['hi']], [y, y], color=color, lw=2, solid_capstyle='round', zorder=2)
        ax.plot([r['nfg']], [y], 'o', color=color, ms=8, zorder=3,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        ypos.append(y)
        labels.append(f'{method} · {side}')
        y += 1
    seps.append(y - 0.5)
    y += 0.45

ax.axvline(0, color=INK, lw=1, zorder=1)
for s in seps[:-1]:
    ax.axhline(s + 0.22, color=GRIDLINE, lw=0.8, zorder=0)

ax.set_yticks(ypos)
ax.set_yticklabels(labels, fontsize=9.5, color=INK)
ax.set_ylim(y - 0.45, -0.7)
ax.set_xlim(-1.08, 1.08)
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_xlabel('normalized fairness gain   (negative = less fair than baseline)',
              fontsize=9.5, color=INK_MUTED)
ax.tick_params(length=0)
ax.grid(axis='x', color=GRIDLINE, lw=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title('Does each stakeholder side improve?   pooled over 4 datasets x 10 seeds',
             fontsize=11.5, color=INK, pad=34, loc='left')
ax.text(0, 1.045, f'point = median NFG, bar = {cov:.0%} order-statistic CI',
        transform=ax.transAxes, fontsize=8.5, color=INK_MUTED, va='bottom')

ax.legend([Line2D([0], [0], color=c, lw=2, marker='o', ms=7, markeredgecolor=SURFACE)
           for c in (IMPROVED, DEGRADED, NO_CHANGE_LINE)],
          ['improves  (q < .05)', 'degrades  (q < .05)', 'no significant change'],
          loc='upper center', bbox_to_anchor=(0.5, -0.11), ncol=3, frameon=False,
          fontsize=9, columnspacing=1.8)

fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(f'{base_path_root}/agg_results/fig_rq4_forest.{ext}',
                dpi=200, bbox_inches='tight')
plt.show()